# 02 Data Cleaning

**Objective:** prepare a clean analysis dataset from `past_rates.csv` without modifying the raw files. Every check and every exclusion is documented, with counts.

**Approach:** work on a copy of the raw data, check first, and only exclude records when there is a clear reason.

import and path

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = PROJECT_ROOT / "data" / "raw"
PROCESSED = PROJECT_ROOT / "data" / "processed"

load the raw data and keep a copy

In [2]:
raw = pd.read_csv(RAW / "past_rates.csv")
df = raw.copy()          # all cleaning happens on this copy; `raw` stays untouched
print("Rows loaded:", len(raw))

# Running log of every exclusion, so the report can quote exact numbers
exclusion_log = []
def log_exclusion(step, reason, n_rows, n_listings=None):
    exclusion_log.append({"step": step, "reason": reason, "rows_affected": n_rows, "listings_affected": n_listings})

Rows loaded: 150034


## 1. Data types and time variables
`month_date` is converted to a datetime and `avg_daily_rate` to a numeric type. Values that cannot be converted would become missing; we count them so nothing is lost silently. We then create `year` and `month` columns.

In [3]:
before_missing = df["avg_daily_rate"].isna().sum()

df["month_date"] = pd.to_datetime(df["month_date"], errors="coerce")
df["avg_daily_rate"] = pd.to_numeric(df["avg_daily_rate"], errors="coerce")

print("month_date missing after conversion:", int(df["month_date"].isna().sum()))
print("avg_daily_rate missing before:", int(before_missing), "| after:", int(df["avg_daily_rate"].isna().sum()))

df["year"] = df["month_date"].dt.year
df["month"] = df["month_date"].dt.month

print()
print(df[["month_date", "year", "month", "avg_daily_rate"]].dtypes.to_string())
print()
print(df[["month_date", "year", "month"]].drop_duplicates().sort_values("month_date").head(3).to_string())

month_date missing after conversion: 0
avg_daily_rate missing before: 0 | after: 0

month_date        datetime64[us]
year                       int32
month                      int32
avg_daily_rate           float64

  month_date  year  month
0 2021-03-01  2021      3
1 2021-04-01  2021      4
2 2021-05-01  2021      5


## 2. Duplicate checks
Two checks: (a) fully identical rows, and (b) repeated `listing_id` + `month_date` pairs. Each listing should appear once per month. Only counting here; nothing is removed.

In [4]:
full_dups = int(df.duplicated().sum())
pair_dups = int(df.duplicated(subset=["listing_id", "month_date"]).sum())

print("Fully identical duplicate rows:", full_dups)
print("Repeated listing_id + month_date pairs:", pair_dups)

per_listing = df.groupby("listing_id").size()
print()
print("Rows per listing (summary):")
print(per_listing.describe().round(1).to_string())
print("Maximum possible months:", df["month_date"].nunique())
print("Listings with more rows than months:", int((per_listing > df["month_date"].nunique()).sum()))

Fully identical duplicate rows: 0
Repeated listing_id + month_date pairs: 0

Rows per listing (summary):
count    4266.0
mean       35.2
std        15.8
min         4.0
25%        21.0
50%        38.0
75%        45.0
max        60.0
Maximum possible months: 60
Listings with more rows than months: 0


## 3. Coordinate validation
Coordinates are checked against NYC's bounding box (taken from the DCP NTA metadata extent), and each listing is checked for a single, consistent location. This only catches gross errors; finer problems (points just off a polygon edge) are examined in the spatial join.

In [5]:
# NYC bounding box from DCP NTA metadata: West, East, South, North
W, E, S, N = -74.257159, -73.699215, 40.496010, 40.915568

print("Rows with missing coordinates:", int(df[["latitude", "longitude"]].isna().any(axis=1).sum()))

outside = df[~df["longitude"].between(W, E) | ~df["latitude"].between(S, N)]
print("Rows outside the NYC bounding box:", len(outside), "| listings:", outside["listing_id"].nunique())
print(outside.groupby("listing_id").agg(rows=("month_date", "size"),
                                        latitude=("latitude", "first"),
                                        longitude=("longitude", "first")).to_string())

loc_check = df.groupby("listing_id")[["latitude", "longitude"]].nunique()
print()
print("Listings with more than one distinct latitude or longitude:", int((loc_check > 1).any(axis=1).sum()))


Rows with missing coordinates: 0
Rows outside the NYC bounding box: 55 | listings: 1
            rows  latitude  longitude
listing_id                           
12762497      55     48.81    1.90756

Listings with more than one distinct latitude or longitude: 0


## 4. Extreme rates and category checks
We inspect the highest and lowest `avg_daily_rate` values together with their context (room type, bedrooms, occupancy). A high value is not automatically an error. This cell only inspects; any exclusion is decided afterwards and documented.

In [6]:
print("room_type:"); print(df["room_type"].value_counts(dropna=False).to_string())
print()
print("listing_type (top 10):"); print(df["listing_type"].value_counts(dropna=False).head(10).to_string())

r = df["avg_daily_rate"]
print()
print("Rate percentiles (USD):"); print(r.quantile([0.5, 0.9, 0.95, 0.99, 0.999]).round(1).to_string())
for t in [1000, 2000, 3000]:
    print(f"Rows above ${t}:", int((r > t).sum()), "| listings:", df.loc[r > t, "listing_id"].nunique())

cols = ["listing_id", "month_date", "avg_daily_rate", "room_type", "listing_type", "bedrooms", "occupancy", "available_days"]
print()
print("10 highest rates:"); print(df.nlargest(10, "avg_daily_rate")[cols].to_string())
print()
print("5 lowest rates:"); print(df.nsmallest(5, "avg_daily_rate")[cols].to_string())

room_type:
room_type
entire_home     84702
private_room    62493
shared_room      2049
hotel_room        766
Hotel room         24

listing_type (top 10):
listing_type
Entire rental unit             59708
Private room in rental unit    34288
Private room in home            9645
Entire home                     6903
Entire condo                    6462
Private room in townhouse       5910
Private room in condo           3394
Room in hotel                   3229
Entire loft                     2966
Entire townhouse                2863

Rate percentiles (USD):
0.500     140.6
0.900     354.4
0.950     476.4
0.990     950.9
0.999    1999.6
Rows above $1000: 1250 | listings: 82
Rows above $2000: 150 | listings: 15
Rows above $3000: 36 | listings: 7

10 highest rates:
       listing_id month_date  avg_daily_rate    room_type        listing_type  bedrooms  occupancy  available_days
4292     39507054 2023-05-01          4983.1  entire_home  Entire rental unit       1.0      0.000              3

In [7]:
r = df["avg_daily_rate"]
print("Percentiles (USD):", r.quantile([0.5, 0.9, 0.95, 0.99, 0.999]).round(0).to_dict())
for t in [1000, 2000, 3000]:
    print(f"Above ${t}: rows={int((r > t).sum())}, listings={df.loc[r > t, 'listing_id'].nunique()}")

zero_occ = df["occupancy"] == 0
print()
print("Rows with occupancy == 0:", int(zero_occ.sum()), f"({100*zero_occ.mean():.1f}%)")
print("Median rate | occupancy == 0:", df.loc[zero_occ, "avg_daily_rate"].median().round(1),
      "| occupancy > 0:", df.loc[~zero_occ, "avg_daily_rate"].median().round(1))

cols = ["listing_id", "month_date", "avg_daily_rate", "room_type", "bedrooms", "occupancy"]
print()
print("8 highest rates:")
print(df.nlargest(8, "avg_daily_rate")[cols].to_string(index=False))
print()
print("Listings with the highest median rate:")
print(df.groupby("listing_id")["avg_daily_rate"].median().nlargest(5).round(0).to_string())

Percentiles (USD): {0.5: 141.0, 0.9: 354.0, 0.95: 476.0, 0.99: 951.0, 0.999: 2000.0}
Above $1000: rows=1250, listings=82
Above $2000: rows=150, listings=15
Above $3000: rows=36, listings=7

Rows with occupancy == 0: 77152 (51.4%)
Median rate | occupancy == 0: 137.4 | occupancy > 0: 144.2

8 highest rates:
 listing_id month_date  avg_daily_rate   room_type  bedrooms  occupancy
   39507054 2023-05-01          4983.1 entire_home       1.0      0.000
   39507054 2023-04-01          4980.3 entire_home       1.0      0.000
   39507054 2023-03-01          4915.0 entire_home       1.0      0.000
    6702693 2022-04-01          4321.9 entire_home       4.0      0.400
    6702693 2022-03-01          4152.6 entire_home       4.0      1.000
    6702693 2022-02-01          4034.8 entire_home       4.0      0.821
    6702693 2022-01-01          4012.2 entire_home       4.0      0.290
    6702693 2021-12-01          3951.7 entire_home       4.0      0.548

Listings with the highest median rate:
listi

In [8]:
# Standardise room_type labels
df["room_type"] = df["room_type"].replace({"Hotel room": "hotel_room"})
print(df["room_type"].value_counts().to_string())

# Flag rows whose coordinates fall outside the NYC bounding box (kept in the data, not deleted)
W, E, S, N = -74.257159, -73.699215, 40.496010, 40.915568
df["coords_in_nyc_box"] = df["longitude"].between(W, E) & df["latitude"].between(S, N)

n_rows = int((~df["coords_in_nyc_box"]).sum())
n_list = df.loc[~df["coords_in_nyc_box"], "listing_id"].nunique()
log_exclusion("coordinates", "Coordinates outside NYC bounding box; excluded from SPATIAL analysis only (kept for citywide temporal analysis)", n_rows, n_list)
print()
print(pd.DataFrame(exclusion_log).to_string(index=False))

room_type
entire_home     84702
private_room    62493
shared_room      2049
hotel_room        790

       step                                                                                                          reason  rows_affected  listings_affected
coordinates Coordinates outside NYC bounding box; excluded from SPATIAL analysis only (kept for citywide temporal analysis)             55                  1


## 5. Neighbourhood assignment (spatial join)
The Airbnb `neighborhood` field holds ZIP-code-like values that do not match NTA names, so each listing is assigned to a 2020 NTA polygon by a point-in-polygon join on its coordinates. Rows in non-residential NTAs (parks, cemeteries, special areas) or outside all polygons are flagged and excluded from the SPATIAL analysis only; they remain in the data for the citywide temporal analysis.

In [9]:
import geopandas as gpd

nta = gpd.read_file(RAW / "nta2020.geojson")[["nta2020", "ntaname", "ntatype", "boroname", "geometry"]]

loc = df.drop_duplicates("listing_id")[["listing_id", "latitude", "longitude"]]
pts = gpd.GeoDataFrame(loc, geometry=gpd.points_from_xy(loc["longitude"], loc["latitude"]), crs="EPSG:4326")
joined = gpd.sjoin(pts, nta, how="left", predicate="within")
assert len(joined) == len(pts) and joined["listing_id"].is_unique   # one polygon per listing

n_before = len(df)
df = df.merge(joined[["listing_id", "nta2020", "ntaname", "ntatype", "boroname"]], on="listing_id", how="left")
assert len(df) == n_before   # the merge must not add or drop rows

df["use_in_spatial"] = df["coords_in_nyc_box"] & (df["ntatype"] == "0")

# Log the spatial exclusions (listings and rows)
no_poly = df["coords_in_nyc_box"] & df["nta2020"].isna()
non_res = df["ntatype"].notna() & (df["ntatype"] != "0")
log_exclusion("spatial", "Inside NYC box but outside every NTA polygon (spatial analysis only)", int(no_poly.sum()), df.loc[no_poly, "listing_id"].nunique())
log_exclusion("spatial", "Inside a non-residential NTA (park, cemetery, special area, airport, Rikers) (spatial analysis only)", int(non_res.sum()), df.loc[non_res, "listing_id"].nunique())

print("Rows:", len(df), "| Listings:", df["listing_id"].nunique())
print("Rows usable for spatial analysis:", int(df["use_in_spatial"].sum()), "| listings:", df.loc[df["use_in_spatial"], "listing_id"].nunique())
print()
print(pd.DataFrame(exclusion_log).to_string(index=False))

Rows: 150034 | Listings: 4266
Rows usable for spatial analysis: 148576 | listings: 4221

       step                                                                                                          reason  rows_affected  listings_affected
coordinates Coordinates outside NYC bounding box; excluded from SPATIAL analysis only (kept for citywide temporal analysis)             55                  1
    spatial                                            Inside NYC box but outside every NTA polygon (spatial analysis only)             45                  2
    spatial            Inside a non-residential NTA (park, cemetery, special area, airport, Rikers) (spatial analysis only)           1358                 42


In [10]:
PROCESSED.mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / "outputs" / "tables").mkdir(parents=True, exist_ok=True)

df.to_parquet(PROCESSED / "past_rates_clean.parquet", index=False)     # local only (ignored by Git)
pd.DataFrame(exclusion_log).to_csv(PROJECT_ROOT / "outputs" / "tables" / "02_exclusion_log.csv", index=False)

print("Saved:", (PROCESSED / "past_rates_clean.parquet").name, "|", len(df), "rows,", df.shape[1], "columns")

Saved: past_rates_clean.parquet | 150034 rows, 31 columns


## Summary
- Starting data: 150,034 rows (listing-months) from 4,266 listings. The raw files were not modified; all cleaning was done on an in-memory copy.
- `month_date` and `avg_daily_rate` converted without loss (0 missing). `year` and `month` columns created.
- No duplicate rows and no repeated listing-month pairs. Listings have between 4 and 60 monthly records, so the panel is unbalanced.
- All rates are in USD. No listing changes location over time.
- `room_type`: the labels `hotel_room` and `Hotel room` were merged (790 rows).
- One listing (ID 12762497, 55 rows) has coordinates in France. It is excluded from spatial analysis only.
- Neighbourhoods were assigned by a point-in-polygon join to the 2020 NTAs. Excluded from spatial analysis only: 45 rows (2 listings) outside every polygon and 1,358 rows (42 listings) inside non-residential NTAs.
- Rows usable for spatial analysis: 148,576 (4,221 listings). All 150,034 rows remain available for the citywide temporal analysis.
- No rates were removed.

## Limitations / observations
- 1,250 rows (82 listings) have `avg_daily_rate` above $1,000 and 36 rows (7 listings) above $3,000. These were retained because there is no evidence they are errors; the median-based method limits their influence.
- 51.4% of rows have occupancy of 0 but still report a rate; the rate in those months may not reflect a realised booking price. A sensitivity check using only months with occupancy above 0 is planned.
- Listing-level counts per NTA are uneven; a minimum-listings rule will be chosen separately for each year in the spatial analysis.
- The Airbnb `neighborhood` field (ZIP-like codes) was not used for the spatial analysis.